In [1]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r /kaggle/input/isic2024-src /kaggle/working; mv /kaggle/working/isic2024-src /kaggle/working/src
!ln -s /kaggle/input/isic2024-configs /kaggle/working/configs

!mkdir -p /kaggle/working/logs/train/runs

!mkdir -p /kaggle/working/logs/train_all_data/runs
!ln -s /kaggle/input/train-all-data-0821-beitv2-base/0821-beitv2_base-sep_head-transV8-lr1e-3-target_decay001-warmup50-wd1e-2-bs32_8-mixup-ep100-neg3-tsgkf /kaggle/working/logs/train_all_data/runs/
!ln -s /kaggle/input/train-all-data-0821-convnextv2-tiny-meta-t03-tsgkf/0821-convnextv2_tiny-meta_target03-transV2-lr1e-3-target_decay001-bs128_2-ep100-neg5-tsgkf /kaggle/working/logs/train_all_data/runs/
!ln -s /kaggle/input/train-all-data-0821-eva02-small/0821-eva02_small-sep_head-transV6-lr1e-3-target_decay001-warmup50-wd1e-2-drop01-bs32_8-ep80-neg3-tsgkf /kaggle/working/logs/train_all_data/runs/
!ln -s /kaggle/input/train-all-data-0824-eva02-small/0824-eva02_small-sep_head-transV8-lr1e-3-target_decay0008-warmup50-wd1e-2-drop01-bs32_8-ep80-neg3-cluster7t5-tsgkf /kaggle/working/logs/train_all_data/runs/
!ln -s /kaggle/input/train-all-data-0824-swinv2-small/0824-swinv2_small-transV2-lr1e-3-target_decay001-bs32_8-drop01-ep200-neg3-cluster7t5-tsgkf /kaggle/working/logs/train_all_data/runs/
!ln -s /kaggle/input/train-all-data-0825-tip-onlyimage-cnv2-n-bs64-2/0825-tip_finetune_onlyImage-convnextv2_nano_scratch_l2d192h8-tabV3-transV8-lr5e-4-warmup50-bs_64_8-neg50-ep200-tsgkf-lr1e-3-warmup5-bs64_2-transV2-ep80 /kaggle/working/logs/train_all_data/runs/
!ln -s /kaggle/input/train-all-data-0827-deit3-small/0827-deit3_small-transV2-lr1e-3-target_decay001-bs32_8-ep200-neg3-tsgkf /kaggle/working/logs/train_all_data/runs/
!ln -s /kaggle/input/train-all-data-0828-resnext50/0828-resnext50-transV2-lr1e-3-target_decay001-bs32_8-ep200-neg3-cluster7t5-tsgkf /kaggle/working/logs/train_all_data/runs/
!ln -s /kaggle/input/train-all-data-0827-tip-onlyimage-swin-tiny-bs64-2/0827-tip_finetune_onlyImage-swin_tiny_scratch_l2d192h8-tabV3-transV8-lr5e-4-warmup50-bs_64_8-neg50-ep200-tsgkf-lr1e-3-warmup5-bs_64_2-transV8-ep80 /kaggle/working/logs/train_all_data/runs/
!mkdir -p /kaggle/working/logs/gbdt/runs
!ln -s /kaggle/input/0906-9nns-18types-fev7-s5/0906-9NNs-18types-feV7-s5 /kaggle/working/logs/gbdt/runs/
!ln -s /kaggle/input/0906-9nns-18types-fev7-s5-tuning-weights/0906-9NNs-18types-feV7-s5-tuning_weights /kaggle/working/logs/gbdt/runs/

!touch /kaggle/working/.project-root

!mkdir /kaggle/working/data
!ln -s /kaggle/input/isic-2024-challenge /kaggle/working/data/isic-2024-challenge
gbdt_params = "0906-9NNs-18types-feV7-s5-tuning_weights"
batch_size_pred=128

DEBUG_WITH_TRAIN_DATA = False
average = "simple"
use_model_dnn = "all_data"
use_model_gbdt = "all_data"
use_shina_models = False
assert average in ["simple", "rank"]
assert use_model_dnn in ["kfold", "all_data"]
assert use_model_gbdt in ["kfold", "all_data"]

!pip install -r /content/drive/MyDrive/isic_2024_second_place/input/isic2024-pip/requirements.txt
!pip install hydra-core lightning catboost rootutils
import hydra
import rootutils
import joblib
import os
from lightning import LightningDataModule, LightningModule, Trainer
from hydra import compose, initialize
from pathlib import Path
from glob import glob
import numpy as np
import torch
import pandas as pd
import lightgbm as lgb
import catboost as cb
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
os.chdir("/content/drive/MyDrive/isic_2024_second_place/working/src")
rootutils.setup_root(Path().resolve(), indicator=".project-root", pythonpath=True)
from src.utils import RankedLogger
from src.isic_utils.feature_engineering import feature_engineering
from src.isic_utils.utils import comp_score, preprocess_df
from src.isic_utils.feature_engineering import feature_engineering_new
from src.isic_utils.utils import comp_score, preprocess_df
from src.isic_utils.feature_engineering import feature_engineering_new
from src.isic_utils.gbdt_models import GBDTModels
root = os.environ["PROJECT_ROOT"]
log = RankedLogger(__name__, rank_zero_only=True)

from hydra import compose, initialize_config_dir
with initialize_config_dir(version_base=None, config_dir="/content/drive/MyDrive/isic_2024_second_place/working/configs"):
    cfg = compose(
        config_name="gbdt",
        overrides=[f"gbdt_params={gbdt_params}"],
        return_hydra_config=True,
    )
    cfg.paths.output_dir = "${hydra.runtime.output_dir}"
    cfg.paths.work_dir = "${hydra.runtime.cwd}"
    cfg.hydra.run.dir = cfg.log_dir
    cfg.hydra.runtime.output_dir = cfg.hydra.run.dir
if DEBUG_WITH_TRAIN_DATA:
    cfg.data.hdf5_test_name="train-image.hdf5"
    cfg.data.meta_csv_test_name="train-metadata.csv"

df_train = pd.read_csv(os.path.join(cfg.data.data_dir, cfg.data.meta_csv_train_name))
df_test = pd.read_csv(os.path.join(cfg.data.data_dir, cfg.data.meta_csv_test_name))
df_train, feature_cols, cat_cols = feature_engineering_new(df_train, version=cfg.gbdt_params.version_fe)
df_test, _, _ = feature_engineering_new(df_test, version=cfg.gbdt_params.version_fe)
df_train, df_test, feature_cols, cat_cols = preprocess_df(df_train, df_test, feature_cols, cat_cols)
from torch.utils.data import DataLoader, Dataset
import tempfile
from tqdm.notebook import tqdm
import gc

def strip_orig_mod_prefix(state_dict):
    new_state_dict = {}
    for key, value in state_dict.items():
        new_key = key.replace("net._orig_mod.", "net.")
        new_state_dict[new_key] = value
    return new_state_dict
def inference(temp_dir, cfg):
    print("Running inference on single device.")
    for dnn_run in cfg.gbdt_params.dnn_predictions:
        config_path = os.path.join("/content/drive/MyDrive/isic_2024_second_place/working/configs/experiment", f"{dnn_run.name}.yaml")
        if not os.path.exists(config_path):
            print(f"Configuration file for {dnn_run.name} not found at {config_path}. Skipping this experiment.")
            continue
        with initialize_config_dir(version_base=None, config_dir="/content/drive/MyDrive/isic_2024_second_place/working/configs"):
            cfg_dnn = compose(
                config_name="train",
                overrides=[f"experiment={dnn_run.name}"],
                return_hydra_config=True,
            )
            cfg_dnn.paths.output_dir = "${hydra.runtime.output_dir}"
            cfg_dnn.paths.work_dir = "${hydra.runtime.cwd}"
            cfg_dnn.hydra.run.dir = cfg_dnn.log_dir
            cfg_dnn.hydra.runtime.output_dir = cfg_dnn.hydra.run.dir
        if cfg_dnn.model.net.get("pretrained"):
            cfg_dnn.model.net.pretrained=False
        if cfg_dnn.model.net.get("my_pretrain_path"):
            cfg_dnn.model.net.my_pretrain_path=None
        if cfg_dnn.model.net.get("image_model"):
            if cfg_dnn.model.net.image_model.get("pretrained"):
                cfg_dnn.model.net.image_model.pretrained=False
            if cfg_dnn.model.net.image_model.get("my_pretrain_path"):
                cfg_dnn.model.net.image_model.my_pretrain_path=None
        if cfg_dnn.model.net.get("meta_pretrain_path"):
            cfg_dnn.model.net.meta_pretrain_path=None
        if cfg_dnn.model.net.get("image_pretrain_path"):
            cfg_dnn.model.net.image_pretrain_path=None
        if cfg_dnn.model.net.get("ckpt_path"):
            cfg_dnn.model.net.ckpt_path=None
        if cfg_dnn.model.net.get("encoder_image"):
            cfg_dnn.model.net.encoder_image.pretrained=False
        cfg_dnn.data.num_workers = 2
        if DEBUG_WITH_TRAIN_DATA:
            cfg_dnn.data.hdf5_test_name="train-image.hdf5"
            cfg_dnn.data.meta_csv_test_name="train-metadata.csv"
        datamodule: LightningDataModule = hydra.utils.instantiate(cfg_dnn.data)
        datamodule.setup("predict")
        dataset = datamodule.data_pred
        dataloader = DataLoader(dataset, batch_size=batch_size_pred, num_workers=2)
        ckpt_paths = []
        for fold in range(cfg_dnn.data.n_fold):
            if fold == 0:
                ckpt_name = "last.ckpt"
            else:
                ckpt_name = f"last-v{fold}.ckpt"
            ckpt_paths.append(glob(os.path.join(cfg_dnn.paths.output_dir, "checkpoints", ckpt_name))[0])
        model: LightningDataModule = hydra.utils.instantiate(cfg_dnn.model)
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        model = model.to(device).eval()
        cfg_dnn.model.compile = False
        if cfg_dnn.model.compile:
            model.net = torch.compile(model.net)
        for fold, ckpt_path in enumerate(ckpt_paths):
            state_dict = torch.load(ckpt_path, map_location=device)["state_dict"]
            state_dict = strip_orig_mod_prefix(state_dict)
            model.load_state_dict(state_dict)
            net = model.net
            net.eval()
            all_predictions = []
            ids = []
            with torch.inference_mode():
                with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                    for data in tqdm(dataloader, desc=f"fold_{fold}"):
                        if cfg_dnn.model._target_ == "src.models.isic2024_module_tip_finetune.ISIC2024LitModuleTIPFinetune":
                            logits = net(data["image"].to(device), data["tabular"].to(device))
                        else:
                            if cfg_dnn.model.get("use_image") and cfg_dnn.model.get("use_metadata"):
                                logits = net(data["image"].to(device), data["metadata_num"].to(device), data["metadata_cat"].to(device))
                            else:
                                logits = net(data["image"].to(device))
                        if cfg.gbdt_params.use_logits:
                            preds = logits[:, 1]
                        else:
                            preds = torch.softmax(logits, dim=1)[:, 1]
                        all_predictions.append(preds.cpu())
                        ids.append(data["isic_id"])
            all_predictions = torch.cat(all_predictions)
            ids = np.concatenate(ids)
            temp_file = os.path.join(temp_dir, f"predictions_{dnn_run.name}_fold_{fold}.pt")
            torch.save(all_predictions, temp_file)
            temp_file_id = os.path.join(temp_dir, f"ids_{dnn_run.name}_fold_{fold}.npy")
            np.save(temp_file_id, ids)
            print(f"{dnn_run.name}, fold {fold} finished inference with predictions saved to {temp_file}, {temp_file_id}")
        del all_predictions, ids, datamodule, dataset, model, net, dataloader
        gc.collect()
def inference_mp(cfg):
    temp_dir = "/content/drive/MyDrive/isic_2024_second_place/tmp/inference_temp"
    print(f"Temporary directory for storing predictions: {temp_dir}")
    inference(temp_dir, cfg)
    dnn_run_name_list = []
    df_dnn_preds_list = []
    for dnn_run in cfg.gbdt_params.dnn_predictions:
        config_path = os.path.join("/content/drive/MyDrive/isic_2024_second_place/working/configs/experiment", f"{dnn_run.name}.yaml")
        if not os.path.exists(config_path):
            print(f"Configuration file for {dnn_run.name} not found at {config_path}. Skipping this experiment.")
            continue
        df_preds_list = []
        for fold in range(cfg.data.n_fold):
            temp_file = os.path.join(temp_dir, f"predictions_{dnn_run.name}_fold_{fold}.pt")
            temp_file_id = os.path.join(temp_dir, f"ids_{dnn_run.name}_fold_{fold}.npy")
            if not os.path.exists(temp_file) or not os.path.exists(temp_file_id):
                print(f"Prediction files for {dnn_run.name} fold {fold} not found at {temp_file} or {temp_file_id}. Skipping this fold.")
                continue
            try:
                predictions = torch.load(temp_file)
                ids = np.load(temp_file_id)
                if not isinstance(predictions, torch.Tensor) or not isinstance(ids, np.ndarray):
                    print(f"Invalid data format for {dnn_run.name} fold {fold} at {temp_file} or {temp_file_id}. Skipping this fold.")
                    continue
                if len(predictions) == 0 or len(ids) == 0:
                    print(f"Empty predictions or IDs for {dnn_run.name} fold {fold} at {temp_file} or {temp_file_id}. Skipping this fold.")
                    continue
                if len(predictions) != len(ids):
                    print(f"Mismatch in lengths of predictions ({len(predictions)}) and IDs ({len(ids)}) for {dnn_run.name} fold {fold}. Skipping this fold.")
                    continue
                df_preds = pd.DataFrame({"isic_id": ids, "target": predictions})
                df_preds = df_preds.drop_duplicates(subset=["isic_id"])
                df_preds_list.append(df_preds)
            except Exception as e:
                print(f"Error loading or processing predictions for {dnn_run.name} fold {fold}: {str(e)}. Skipping this fold.")
                continue
        if df_preds_list:
            df_dnn_preds_list.append(concat_df_preds(df_preds_list))
            dnn_run_name_list.append(dnn_run.name)
    return df_dnn_preds_list, dnn_run_name_list
def inference_all_data(temp_dir, cfg):
    print("Running inference on single device.")
    for dnn_run in cfg.gbdt_params.dnn_predictions:
        config_path = os.path.join("/content/drive/MyDrive/isic_2024_second_place/working/configs/experiment", f"{dnn_run.name}.yaml")
        if not os.path.exists(config_path):
            print(f"Configuration file for {dnn_run.name} not found at {config_path}. Skipping this experiment.")
            continue
        with initialize_config_dir(version_base=None, config_dir="/content/drive/MyDrive/isic_2024_second_place/working/configs"):
            cfg_dnn = compose(
                config_name="train",
                overrides=[f"experiment={dnn_run.name}"],
                return_hydra_config=True,
            )
            cfg_dnn.task_name = "train_all_data"
            cfg_dnn.paths.output_dir = "${hydra.runtime.output_dir}"
            cfg_dnn.paths.work_dir = "${hydra.runtime.cwd}"
            cfg_dnn.hydra.run.dir = cfg_dnn.log_dir
            cfg_dnn.hydra.runtime.output_dir = cfg_dnn.hydra.run.dir
        if cfg_dnn.model.net.get("pretrained"):
            cfg_dnn.model.net.pretrained=False
        if cfg_dnn.model.net.get("my_pretrain_path"):
            cfg_dnn.model.net.my_pretrain_path=None
        if cfg_dnn.model.net.get("image_model"):
            if cfg_dnn.model.net.image_model.get("pretrained"):
                cfg_dnn.model.net.image_model.pretrained=False
            if cfg_dnn.model.net.image_model.get("my_pretrain_path"):
                cfg_dnn.model.net.image_model.my_pretrain_path=None
        if cfg_dnn.model.net.get("meta_pretrain_path"):
            cfg_dnn.model.net.meta_pretrain_path=None
        if cfg_dnn.model.net.get("image_pretrain_path"):
            cfg_dnn.model.net.image_pretrain_path=None
        if cfg_dnn.model.net.get("ckpt_path"):
            cfg_dnn.model.net.ckpt_path=None
        if cfg_dnn.model.net.get("encoder_image"):
            cfg_dnn.model.net.encoder_image.pretrained=False
        cfg_dnn.data.num_workers = 2
        if DEBUG_WITH_TRAIN_DATA:
            cfg_dnn.data.hdf5_test_name="train-image.hdf5"
            cfg_dnn.data.meta_csv_test_name="train-metadata.csv"
        datamodule: LightningDataModule = hydra.utils.instantiate(cfg_dnn.data)
        datamodule.setup("predict")
        dataset = datamodule.data_pred
        dataloader = DataLoader(dataset, batch_size=batch_size_pred, num_workers=2)
        ckpt_name = "last.ckpt"
        ckpt_dir = os.path.join(cfg_dnn.paths.output_dir, "checkpoints")

        if not os.path.exists(ckpt_dir):
            print(f"WARNING: Checkpoint directory does not exist: {ckpt_dir}. Skipping {dnn_run.name}")
            continue

        ckpt_pattern = os.path.join(ckpt_dir, ckpt_name)
        ckpt_paths = glob(ckpt_pattern)

        if not ckpt_paths:
            print(f"WARNING: No 'last.ckpt' found in {ckpt_dir}. Skipping {dnn_run.name}")
            continue

        ckpt_path = ckpt_paths[0]
        print(f"Loading checkpoint: {ckpt_path}")
        model: LightningDataModule = hydra.utils.instantiate(cfg_dnn.model)
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {device}")
        model = model.to(device).eval()
        cfg_dnn.model.compile = False
        if cfg_dnn.model.compile:
            model.net = torch.compile(model.net)
        state_dict = torch.load(ckpt_path, map_location=device)["state_dict"]
        state_dict = strip_orig_mod_prefix(state_dict)
        model.load_state_dict(state_dict)
        net = model.net
        net.eval()
        all_predictions = []
        ids = []
        with torch.inference_mode():
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                for data in tqdm(dataloader, desc=f"single_device"):
                    if cfg_dnn.model._target_ == "src.models.isic2024_module_tip_finetune.ISIC2024LitModuleTIPFinetune":
                        logits = net(data["image"].to(device), data["tabular"].to(device))
                    else:
                        if cfg_dnn.model.get("use_image") and cfg_dnn.model.get("use_metadata"):
                            logits = net(data["image"].to(device), data["metadata_num"].to(device), data["metadata_cat"].to(device))
                        else:
                            logits = net(data["image"].to(device))
                    if cfg.gbdt_params.use_logits:
                        preds = logits[:, 1]
                    else:
                        preds = torch.softmax(logits, dim=1)[:, 1]
                    all_predictions.append(preds.cpu())
                    ids.append(data["isic_id"])
        all_predictions = torch.cat(all_predictions)
        ids = np.concatenate(ids)
        temp_file = os.path.join(temp_dir, f"predictions_{dnn_run.name}.pt")
        torch.save(all_predictions, temp_file)
        temp_file_id = os.path.join(temp_dir, f"ids_{dnn_run.name}.npy")
        np.save(temp_file_id, ids)
        print(f"{dnn_run.name} finished inference with predictions saved to {temp_file}, {temp_file_id}")
        del all_predictions, ids, datamodule, dataset, model, net, dataloader
        gc.collect()
def inference_mp_all_data(cfg):
    temp_dir = "/content/drive/MyDrive/isic_2024_second_place/tmp/inference_temp"
    print(f"Temporary directory for storing predictions: {temp_dir}")
    inference_all_data(temp_dir, cfg)
    dnn_run_name_list = []
    df_dnn_preds_list = []
    for dnn_run in cfg.gbdt_params.dnn_predictions:
        temp_file = os.path.join(temp_dir, f"predictions_{dnn_run.name}.pt")
        temp_file_id = os.path.join(temp_dir, f"ids_{dnn_run.name}.npy")
        if not os.path.exists(temp_file) or not os.path.exists(temp_file_id):
            print(f"Prediction files for {dnn_run.name} not found at {temp_file} or {temp_file_id}. Skipping this experiment.")
            continue
        try:
            predictions = torch.load(temp_file)
            ids = np.load(temp_file_id)
            if not isinstance(predictions, torch.Tensor) or not isinstance(ids, np.ndarray):
                print(f"Invalid data format for {dnn_run.name} at {temp_file} or {temp_file_id}. Skipping this experiment.")
                continue
            if len(predictions) == 0 or len(ids) == 0:
                print(f"Empty predictions or IDs for {dnn_run.name} at {temp_file} or {temp_file_id}. Skipping this experiment.")
                continue
            if len(predictions) != len(ids):
                print(f"Mismatch in lengths of predictions ({len(predictions)}) and IDs ({len(ids)}) for {dnn_run.name}. Skipping this experiment.")
                continue
            df_preds = pd.DataFrame({"isic_id": ids, "target": predictions})
            df_preds = df_preds.drop_duplicates(subset=["isic_id"])
            df_preds = df_preds.rename({"target": dnn_run.name}, axis="columns")
            df_dnn_preds_list.append(df_preds)
            dnn_run_name_list.append(dnn_run.name)
        except Exception as e:
            print(f"Error loading or processing predictions for {dnn_run.name}: {str(e)}. Skipping this experiment.")
            continue
    return df_dnn_preds_list, dnn_run_name_list
def concat_df_preds(df_preds_list):
    df_preds = df_preds_list[0].rename({"target": "predictions"}, axis="columns")
    for k, _df_preds in enumerate(df_preds_list[1:], start=1):
        df_preds = df_preds.merge(_df_preds.rename({"target": "predictions"}, axis="columns"), how="left", on="isic_id", suffixes=("", f"_{k}"))
    df_preds = df_preds.rename({"predictions": "predictions_0"}, axis="columns")
    return df_preds

DEBUG_WITH_TRAIN_DATA = False
if use_model_dnn == "kfold":
    df_dnn_preds_list_test, dnn_run_name_list = inference_mp(cfg)
elif use_model_dnn == "all_data":
    df_dnn_preds_list_test, dnn_run_name_list = inference_mp_all_data(cfg)

DEBUG_WITH_TRAIN_DATA = True
if use_model_dnn == "kfold":
    df_dnn_preds_list_train, _ = inference_mp(cfg)
elif use_model_dnn == "all_data":
    df_dnn_preds_list_train, _ = inference_mp_all_data(cfg)
DEBUG_WITH_TRAIN_DATA = False
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")

!pip install polars
!pip install optuna
!pip install imblearn
!pip install lightgbm
!pip install catboost
!pip install xgboost
!pip install eli5

import os
from glob import glob
from pathlib import Path
import numpy as np
import pandas as pd
import polars as pl
import plotly.graph_objects as go
from typing import List, Tuple, Dict, Optional
from pydantic import BaseModel
from tqdm import tqdm
from sklearn.neighbors import LocalOutlierFactor
from sklearn.model_selection import cross_val_score, StratifiedGroupKFold
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.metrics import roc_auc_score, silhouette_score
from sklearn.ensemble import VotingClassifier
from sklearn.cluster import KMeans
from sklearn.inspection import permutation_importance
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline
from catboost import CatBoostClassifier, Pool
from matplotlib import pyplot as plt
import lightgbm as lgb
import catboost as cb
import xgboost as xgb
from PIL import Image
from scipy.stats import ttest_ind, ttest_1samp, ttest_rel
from eli5.permutation_importance import get_score_importances
import optuna
from optuna.trial import TrialState
import warnings
warnings.filterwarnings('ignore')
root = Path('/content/drive/MyDrive/isic_2024_first_place/data/original')
train_path = root / 'train-metadata.csv'
test_path = root / 'test-metadata.csv'
subm_path = root / 'sample_submission.csv'
id_col = 'isic_id'
target_col = 'target'
group_col = 'patient_id'
err = 1e-5
sampling_ratio = 0.01
seed = 42

columns_to_drop = [
    'tbp_lv_B', 'tbp_lv_C', 'tbp_lv_H', 'tbp_lv_L', 'tbp_lv_radial_color_std_max', 'tbp_lv_y', 'tbp_lv_z',
    'luminance_contrast', 'lesion_color_difference', 'normalized_lesion_size', 'tbp_lv_norm_border_patient_norm',
    'lesion_color_difference_patient_norm', 'age_normalized_nevi_confidence_2_patient_norm', 'tbp_lv_deltaA',
    'size_age_interaction', 'tbp_lv_deltaB', 'oof_eva_score__cluster', 'hue_contrast_patient_norm__cluster',
    'color_contrast_index_patient_norm', 'tbp_lv_Lext__cluster', 'tbp_lv_Cext_patient_norm', 'oof_eva_score_m',
    'age_approx_patient_norm', 'volume_approximation_3d_patient_norm'
]
def set_seed(random_seed):
    import random
    random.seed(random_seed)
    torch.manual_seed(random_seed)
    np.random.seed(random_seed)
def custom_metric_raw(y_hat, y_true):
    min_tpr = 0.80
    max_fpr = abs(1 - min_tpr)
    v_gt = abs(y_true - 1)
    v_pred = np.array([1.0 - x for x in y_hat])
    partial_auc_scaled = roc_auc_score(v_gt, v_pred, max_fpr=max_fpr)
    partial_auc = 0.5 * max_fpr**2 + (max_fpr - 0.5 * max_fpr**2) / (1.0 - 0.5) * (partial_auc_scaled - 0.5)
    return partial_auc
def custom_metric(estimator, X, y_true):
    y_hat = estimator.predict_proba(X)[:, 1]
    partial_auc = custom_metric_raw(y_hat, y_true)
    return partial_auc
def get_lof_score(df_train, top_lof_features, output_col_name="of", lof_column='patient_id'):
    scalled_array = StandardScaler().fit_transform(df_train[top_lof_features])
    outiler_factors = []
    for tmp_col_v in tqdm(df_train[lof_column].unique()):
        mask = (df_train[lof_column] == tmp_col_v).values
        if sum(mask) < 3:
            continue
        patient_emb = scalled_array[mask]
        clf = LocalOutlierFactor(n_neighbors=min(sum(mask), 30))
        clf.fit_predict(patient_emb)
        outiler_factors.append(pd.DataFrame(
            {"isic_id": df_train[mask].index.values,
             output_col_name: clf.negative_outlier_factor_}))
    outiler_factors = pd.concat(outiler_factors).reset_index(drop=True)
    df_train = df_train.merge(outiler_factors.set_index('isic_id'), how="left", left_index=True, right_index=True)
    df_train[output_col_name] = df_train[output_col_name].fillna(-1)
    return df_train
top_lof_features = [
    'tbp_lv_H', 'hue_contrast', 'age_normalized_nevi_confidence_2', 'tbp_lv_deltaB',
    'color_uniformity', 'tbp_lv_z', 'clin_size_long_diam_mm', 'tbp_lv_y',
    'position_distance_3d', 'hue_contrast', 'tbp_lv_stdLExt', 'mean_hue_difference',
    'age_normalized_nevi_confidence', 'lesion_visibility_score', 'position_distance_3d',
    'tbp_lv_minorAxisMM', 'tbp_lv_Hext'
]

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)
df_train, feature_cols, cat_cols = feature_engineering_new(df_train, version=cfg.gbdt_params.version_fe)
df_test, _, _ = feature_engineering_new(df_test, version=cfg.gbdt_params.version_fe)
df_train, df_test, feature_cols, cat_cols = preprocess_df(df_train, df_test, feature_cols, cat_cols)
df_train = df_train.set_index('isic_id')
df_test = df_test.set_index('isic_id')
df_train['count_per_patient'] = df_train.groupby('patient_id')['patient_id'].transform('count')
df_train['tbp_lv_areaMM2_patient'] = df_train.groupby('patient_id')['tbp_lv_areaMM2'].transform('sum')
df_train['tbp_lv_areaMM2_bp'] = df_train.groupby(['patient_id', 'anatom_site_general'])['tbp_lv_areaMM2'].transform('sum')
df_test['count_per_patient'] = df_test.groupby('patient_id')['patient_id'].transform('count')
df_test['tbp_lv_areaMM2_patient'] = df_test.groupby('patient_id')['tbp_lv_areaMM2'].transform('sum')
df_test['tbp_lv_areaMM2_bp'] = df_test.groupby(['patient_id', 'anatom_site_general'])['tbp_lv_areaMM2'].transform('sum')
df_subm = pd.read_csv(subm_path, index_col=id_col)

for i, run_name in enumerate(dnn_run_name_list):
    if use_model_dnn == "kfold":
        df_dnn_preds_train = df_dnn_preds_list_train[i]
        df_dnn_preds_train[run_name] = df_dnn_preds_train[['predictions_0', 'predictions_1', 'predictions_2', 'predictions_3', 'predictions_4']].mean(1)
        df_dnn_preds_train["isic_id"] = df_dnn_preds_train["isic_id"].astype(str)
        df_train = df_train.merge(df_dnn_preds_train[["isic_id", run_name]], how="left", left_index=True, right_on="isic_id").set_index('isic_id')
        df_dnn_preds_test = df_dnn_preds_list_test[i]
        df_dnn_preds_test[run_name] = df_dnn_preds_test[['predictions_0', 'predictions_1', 'predictions_2', 'predictions_3', 'predictions_4']].mean(1)
        df_dnn_preds_test["isic_id"] = df_dnn_preds_test["isic_id"].astype(str)
        df_test = df_test.merge(df_dnn_preds_test[["isic_id", run_name]], how="left", left_index=True, right_on="isic_id").set_index('isic_id')
    elif use_model_dnn == "all_data":
        df_dnn_preds_list_train[i]["isic_id"] = df_dnn_preds_list_train[i]["isic_id"].astype(str)
        df_train = df_train.merge(df_dnn_preds_list_train[i][["isic_id", run_name]], how="left", left_index=True, right_on="isic_id").set_index('isic_id')
        df_dnn_preds_list_test[i]["isic_id"] = df_dnn_preds_list_test[i]["isic_id"].astype(str)
        df_test = df_test.merge(df_dnn_preds_list_test[i][["isic_id", run_name]], how="left", left_index=True, right_on="isic_id").set_index('isic_id')

feature_cols += dnn_run_name_list

missing_dnn_features = [dnn_run.name for dnn_run in cfg.gbdt_params.dnn_predictions if dnn_run.name not in dnn_run_name_list]
for missing_feature in missing_dnn_features:
    if missing_feature not in df_test.columns:
        df_test[missing_feature] = 0.0
        df_train[missing_feature] = 0.0
        print(f"Added placeholder column for missing DNN feature: {missing_feature}")
df_train = get_lof_score(df_train, top_lof_features, output_col_name="of", lof_column='patient_id')
feature_cols += ['of']
def find_prediction_column(df, possible_names=['tmp_predictions_all', 'tmp_predictions_all__pr', 'predictions', 'score']):
    for col in possible_names:
        if col in df.columns:
            return col
    return None
oof_forecasts = pd.read_parquet('/content/drive/MyDrive/isic_2024_first_place/data/artifacts/oof_forecasts_eva_base.parquet')
print("Columns in oof_forecasts_eva_base.parquet:", oof_forecasts.columns.tolist())
pred_col = find_prediction_column(oof_forecasts)
if pred_col:
    df_train = df_train.merge(
        oof_forecasts[['isic_id', pred_col]].rename(columns={pred_col: 'oof_eva_score'}),
        how="left", on=['isic_id'])
    df_train = df_train.merge(
        df_train.groupby("patient_id").agg(**{
            "oof_eva_score_m": pd.NamedAgg('oof_eva_score', 'mean')
        }).reset_index(), how="left", on=["patient_id"])
    df_train['oof_eva_score_m'] = df_train['oof_eva_score'] / df_train['oof_eva_score_m']
    feature_cols += ['oof_eva_score', 'oof_eva_score_m']
else:
    print("Warning: No suitable prediction column found in oof_forecasts_eva_base.parquet. Skipping oof_eva_score merge.")
    columns_to_drop.extend(['oof_eva_score', 'oof_eva_score_m', 'oof_eva_score__cluster', 'oof_eva_score_m__cluster', 'oof_eva_score_mean_tmp', 'oof_eva_score_mean_tmp__cluster'])
oof_forecasts = pd.read_parquet('/content/drive/MyDrive/isic_2024_first_place/data/artifacts/oof_forecasts_edgenext_base.parquet')
print("Columns in oof_forecasts_edgenext_base.parquet:", oof_forecasts.columns.tolist())
pred_col = find_prediction_column(oof_forecasts)
if pred_col:
    df_train = df_train.merge(
        oof_forecasts[['isic_id', pred_col]].rename(columns={pred_col: 'oof_edgenext_score'}),
        how="left", on=['isic_id'])
    df_train = df_train.merge(
        df_train.groupby("patient_id").agg(**{
            "oof_edgenext_score_m": pd.NamedAgg('oof_edgenext_score', 'mean')
        }).reset_index(), how="left", on=["patient_id"])
    df_train['oof_edgenext_score_m'] = df_train['oof_edgenext_score'] / df_train['oof_edgenext_score_m']
    feature_cols += ['oof_edgenext_score', 'oof_edgenext_score_m']
else:
    print("Warning: No suitable prediction column found in oof_forecasts_edgenext_base.parquet. Skipping oof_edgenext_score merge.")
    columns_to_drop.extend(['oof_edgenext_score', 'oof_edgenext_score_m', 'oof_edgenext_score__cluster', 'oof_edgenext_score_m__cluster', 'oof_edgenext_score_mean_tmp', 'oof_edgenext_score_mean_tmp__cluster'])
if 'oof_eva_score' in df_train.columns and 'oof_edgenext_score' in df_train.columns:
    df_train = df_train.merge(
        df_train.groupby(['attribution', 'tbp_lv_location', 'age_approx']).agg(**{
            "oof_edgenext_score_mean_tmp": pd.NamedAgg('oof_edgenext_score', 'mean'),
            "oof_edgenext_score_std_tmp": pd.NamedAgg('oof_edgenext_score', 'std'),
            "oof_eva_score_mean_tmp": pd.NamedAgg('oof_eva_score', 'mean'),
            "oof_eva_score_std_tmp": pd.NamedAgg('oof_eva_score', 'std')
        }).reset_index(), how="left", on=['attribution', 'tbp_lv_location', 'age_approx'])
    df_train["oof_edgenext_score_mean_tmp"] = (
        df_train["oof_edgenext_score"] - df_train["oof_edgenext_score_mean_tmp"]) / df_train["oof_edgenext_score_std_tmp"]
    df_train["oof_eva_score_mean_tmp"] = (
        df_train["oof_eva_score"] - df_train["oof_eva_score_mean_tmp"]) / df_train["oof_eva_score_std_tmp"]
    feature_cols += ['oof_edgenext_score_mean_tmp', 'oof_eva_score_mean_tmp']
else:
    print("Warning: Skipping oof-based aggregations due to missing oof scores.")
    columns_to_drop.extend(['oof_edgenext_score_mean_tmp', 'oof_eva_score_mean_tmp'])

if 'oof_eva_score' in df_train.columns:
    df_test['oof_eva_score'] = df_train['oof_eva_score'].mean()
    df_test['oof_eva_score_m'] = df_train['oof_eva_score_m'].mean()
    df_test['oof_eva_score_mean_tmp'] = df_train['oof_eva_score_mean_tmp'].mean()
if 'oof_edgenext_score' in df_train.columns:
    df_test['oof_edgenext_score'] = df_train['oof_edgenext_score'].mean()
    df_test['oof_edgenext_score_m'] = df_train['oof_edgenext_score_m'].mean()
    df_test['oof_edgenext_score_mean_tmp'] = df_train['oof_edgenext_score_mean_tmp'].mean()

df_test['of'] = -1

for col in feature_cols:
    if col not in df_test.columns:
        df_test[col] = 0
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_train[top_lof_features])
np.random.seed(42)
kmeans = KMeans(n_clusters=29, random_state=42)
cluster_labels = kmeans.fit_predict(X_scaled)
columns_for_cluster_culculations = [
    "tbp_lv_minorAxisMM", "tbp_lv_norm_border", "tbp_lv_norm_color", "tbp_lv_symm_2axis",
    "tbp_lv_radial_color_std_max", "lesion_visibility_score", "mean_hue_difference", "clin_size_long_diam_mm",
    "color_uniformity", "hue_contrast", "tbp_lv_H",
    *([ "oof_eva_score"] if 'oof_eva_score' in df_train.columns else []),
    *([ "oof_edgenext_score"] if 'oof_edgenext_score' in df_train.columns else []),
    'tbp_lv_Aext', 'tbp_lv_Bext', 'tbp_lv_Cext', 'tbp_lv_Lext',
    'tbp_lv_deltaLBnorm', 'tbp_lv_stdLExt', 'position_distance_3d', 'color_contrast_index',
    'age_normalized_nevi_confidence_2', 'volume_approximation_3d', 'age_approx_patient_norm', 'tbp_lv_H_patient_norm',
    'tbp_lv_Hext_patient_norm', 'tbp_lv_L_patient_norm', 'tbp_lv_stdLExt_patient_norm', 'tbp_lv_symm_2axis_angle_patient_norm',
    'tbp_lv_x_patient_norm', 'tbp_lv_y_patient_norm', 'tbp_lv_z_patient_norm', 'lesion_size_ratio_patient_norm', 'hue_contrast_patient_norm',
    'color_uniformity_patient_norm', 'position_distance_3d_patient_norm', 'size_age_interaction_patient_norm', 'mean_hue_difference_patient_norm',
    'std_dev_contrast_patient_norm', 'lesion_orientation_3d_patient_norm',
    'volume_approximation_3d_patient_norm', 'color_range_patient_norm', 'tbp_lv_areaMM2_patient', 'tbp_lv_areaMM2_bp', 'of',
    *([ "oof_eva_score_m"] if 'oof_eva_score_m' in df_train.columns else []),
    *([ "oof_edgenext_score_m"] if 'oof_edgenext_score_m' in df_train.columns else []),
    *([ "oof_edgenext_score_mean_tmp"] if 'oof_edgenext_score_mean_tmp' in df_train.columns else []),
    *([ "oof_eva_score_mean_tmp"] if 'oof_eva_score_mean_tmp' in df_train.columns else []),
]
df_train['cluster_labels'] = cluster_labels
agg_config = {
    f"{i}__mean": pd.NamedAgg(i, "mean")
        for i in columns_for_cluster_culculations
}
agg_config.update({
    f"{i}__std": pd.NamedAgg(i, "std")
        for i in columns_for_cluster_culculations
})
df_train = df_train.merge(
    df_train.groupby("cluster_labels").agg(**agg_config), how="left", on="cluster_labels")
for col in columns_for_cluster_culculations:
    df_train[f"{col}__cluster"] = (df_train[col] - df_train[f'{col}__mean']) / df_train[f'{col}__std']
    feature_cols += [f"{col}__cluster"]
feature_cols = [col for col in feature_cols if col in df_train.columns]
X_test_scaled = scaler.transform(df_test[top_lof_features])
cluster_labels_test = kmeans.predict(X_test_scaled)
df_test['cluster_labels'] = cluster_labels_test
cluster_agg = df_train.groupby("cluster_labels").agg(**agg_config).reset_index()
df_test = df_test.merge(cluster_agg, how="left", on="cluster_labels")
for col in columns_for_cluster_culculations:
    df_test[f"{col}__cluster"] = (df_test[col] - df_test[f'{col}__mean']) / df_test[f'{col}__std']
for col in feature_cols:
    if col not in df_test.columns:
        df_test[col] = 0.0
def run_model_old(cb_params, reduce=True, n_rounds=10, columns_to_drop=None):
    metric_list = []
    columns_to_drop = [] if columns_to_drop is None else columns_to_drop
    for random_seed in range(1, n_rounds):
        tsp = StratifiedGroupKFold(5, shuffle=True, random_state=random_seed)
        metrics_ev_df = []
        test_forecast = []
        val_forecast = []
        for fold_n, (train_index, val_index) in enumerate(tsp.split(df_train, y=df_train.target, groups=df_train[group_col])):
            train_slice_x = df_train.iloc[train_index][[i for i in feature_cols if i not in columns_to_drop]].reset_index(drop=True)
            val_slice_x = df_train.iloc[val_index][[i for i in feature_cols if i not in columns_to_drop]].reset_index(drop=True)
            train_slice_y = df_train.iloc[train_index]['target'].reset_index(drop=True)
            val_slice_y = df_train.iloc[val_index]['target'].reset_index(drop=True)
            cb_model = Pipeline([
                ('sampler_1', RandomOverSampler(sampling_strategy=0.003, random_state=seed)),
                ('sampler_2', RandomUnderSampler(sampling_strategy=sampling_ratio, random_state=seed)),
                ('classifier', cb.CatBoostClassifier(**cb_params)),
            ])
            cb_model.fit(train_slice_x, train_slice_y)
            preds_cb = cb_model.predict_proba(val_slice_x)[:, 1]
            metric = custom_metric_raw(preds_cb, val_slice_y.values)
            metric_list.append(metric)
    if reduce:
        return np.mean(metric_list)
    else:
        return metric_list
class ModelConfigCB(BaseModel):
    iterations: int = 1000
    learning_rate: float = 0.06936242010150652
    l2_leaf_reg: float = 6.216113851699493
    loss_function: str = "Logloss"
    bagging_temperature: float = 1
    random_seed: int = seed
    border_count: int = 128
    grow_policy: str = "SymmetricTree"
    min_data_in_leaf: int = 24
    do_sample: bool = True
    random_strength: float = 1.0
    depth: int = 6
model_config_cb = ModelConfigCB()
def optimise_catboost(trial: optuna.Trial, test=False, reduce=True, columns_to_drop=None, n_rounds=10, cat_cols=None, get_fe=False):
    cat_cols = [] if cat_cols is None else cat_cols
    columns_to_drop = [] if columns_to_drop is None else columns_to_drop
    model_config_cb = ModelConfigCB(
        iterations=2000,
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.08) if not test else trial.get('learning_rate'),
        l2_leaf_reg=trial.suggest_float('l2_leaf_reg', 1, 20) if not test else trial.get('l2_leaf_reg'),
        random_strength=trial.suggest_float('random_strength', 0, 5) if not test else trial.get('random_strength'),
        loss_function="Logloss",
        depth=trial.suggest_int('depth', 2, 8) if not test else trial.get('depth'),
        bagging_temperature=trial.suggest_float('bagging_temperature', 0, 10) if not test else trial.get('bagging_temperature'),
        border_count=trial.suggest_categorical('border_count', [128, 256]) if not test else trial.get('border_count'),
        grow_policy=trial.suggest_categorical('grow_policy', ["SymmetricTree", "Depthwise", "Lossguide"]) if not test else trial.get('grow_policy'),
        random_seed=42,
        min_data_in_leaf=trial.suggest_int('min_data_in_leaf', 8, 40) if not test else trial.get('min_data_in_leaf'),
    )
    metric_list = []
    models = []
    all_val_df_tmp = []
    score_decreases_list = []
    for random_seed in range(1, n_rounds):
        tsp = StratifiedGroupKFold(5, shuffle=True, random_state=random_seed)
        metrics_ev_df = []
        test_forecast = []
        val_forecast = []
        for fold_n, (train_index, val_index) in enumerate(tsp.split(df_train, y=df_train.target, groups=df_train[group_col])):
            train_slice_x = df_train.iloc[train_index][[i for i in feature_cols if i not in columns_to_drop]].reset_index(drop=True)
            val_slice_x = df_train.iloc[val_index][[i for i in feature_cols if i not in columns_to_drop]].reset_index(drop=True)
            train_slice_y = df_train.iloc[train_index]['target'].reset_index(drop=True)
            val_slice_y = df_train.iloc[val_index]['target'].reset_index(drop=True)
            if model_config_cb.do_sample:
                cb_model = Pipeline([
                    ('sampler_1', RandomOverSampler(sampling_strategy=0.003, random_state=random_seed)),
                    ('sampler_2', RandomUnderSampler(sampling_strategy=sampling_ratio, random_state=random_seed)),
                ])
                train_slice_x, train_slice_y = cb_model.fit_resample(train_slice_x, train_slice_y)
            clf_catboost = CatBoostClassifier(
                loss_function=model_config_cb.loss_function,
                eval_metric='AUC',
                task_type='GPU',
                learning_rate=model_config_cb.learning_rate,
                od_wait=100,
                random_state=random_seed,
                depth=model_config_cb.depth,
                l2_leaf_reg=model_config_cb.l2_leaf_reg,
                min_data_in_leaf=model_config_cb.min_data_in_leaf,
                bagging_temperature=model_config_cb.bagging_temperature,
                border_count=model_config_cb.border_count,
                grow_policy=model_config_cb.grow_policy,
                devices='0',
                iterations=model_config_cb.iterations,
            )
            train_pool = Pool(train_slice_x, train_slice_y.values, cat_features=cat_cols)
            val_pool = Pool(val_slice_x, val_slice_y.values, cat_features=cat_cols)
            clf_catboost.fit(train_pool, eval_set=val_pool, verbose=False)
            preds_cb = clf_catboost.predict_proba(val_slice_x)[:, 1]
            metric = custom_metric_raw(preds_cb, val_slice_y.values)
            metric_list.append(metric)
            models.append(clf_catboost)
            if get_fe:
                score_decreases = permutation_importance(clf_catboost, val_slice_x, val_slice_y.values,
                                                       n_repeats=4,
                                                       random_state=0)
                score_decreases_list.append(score_decreases)
            val_df_tmp = df_train.iloc[val_index].reset_index(drop=False)
            val_df_tmp['predictions'] = preds_cb
            val_df_tmp['random_seed'] = random_seed
            all_val_df_tmp.append(val_df_tmp)
    if reduce:
        return np.mean(metric_list), models, pd.concat(all_val_df_tmp).reset_index(drop=True), score_decreases_list
    else:
        return metric_list, models, pd.concat(all_val_df_tmp).reset_index(drop=True), score_decreases_list
model_config_cb = ModelConfigCB(**{
    'learning_rate': 0.02606161517843435,
    'l2_leaf_reg': 18.04422276698195,
    'random_strength': 4.7069580783889995,
    'depth': 6,
    'bagging_temperature': 0.8735940473548339,
    'border_count': 256,
    'grow_policy': 'Lossguide',
    'min_data_in_leaf': 38
})
kaggle_model_config = {
    'loss_function': 'Logloss',
    'iterations': 200,
    'verbose': False,
    'random_state': seed,
    'max_depth': 7,
    'learning_rate': 0.06936242010150652,
    'scale_pos_weight': 2.6149345838209532,
    'l2_leaf_reg': 6.216113851699493,
    'subsample': 0.6249261779711819,
    'min_data_in_leaf': 24,
    'cat_features': cat_cols,
}
scores_base, models, oof_values, fe_raw = optimise_catboost(model_config_cb.dict(), test=True, reduce=False,
    columns_to_drop=columns_to_drop,
    cat_cols=cat_cols, get_fe=False)
print(np.mean(scores_base))


Mounted at /content/drive
cp: cannot stat '/kaggle/input/isic2024-src': No such file or directory
mv: cannot stat '/kaggle/working/isic2024-src': No such file or directory
ln: failed to create symbolic link '/kaggle/working/configs': No such file or directory
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.9/780.9 MB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 155.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.2/164.2 MB 15.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 117.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 57.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 146.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7

/tmp/ipykernel_2012/581454812.py:83: DtypeWarning: Columns (51,52) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv(os.path.join(cfg.data.data_dir, cfg.data.meta_csv_train_name))


Analyzing ducklings

串流輸出內容已截斷至最後 5000 行。
/content/drive/MyDrive/isic_2024_second_place/working/src/isic_utils/feature_engineering.py:2908: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  z_scores = group[ud_columns].apply(lambda x: zscore(x, nan_policy="omit"))
/content/drive/MyDrive/isic_2024_second_place/working/src/isic_utils/feature_engineering.py:2908: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  z_scores = group[ud_columns].apply(lambda x: zscore(x, nan_policy="omit"))
/content/drive/MyDrive/isic_2024_second_place/working/src/isic_utils/feature_engineering.py:2908: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  z_scores = group


Concat ducklings
Extending ducklings
Enhancing ugly duckling features........................................................................................................
Analyzing ducklings

...

0828-resnext50-transV2-lr1e-3-target_decay001-bs32_8-ep200-neg3-cluster7t5-tsgkf finished inference with predictions saved to /content/drive/MyDrive/isic_2024_second_place/tmp/inference_temp/predictions_0828-resnext50-transV2-lr1e-3-target_decay001-bs32_8-ep200-neg3-cluster7t5-tsgkf.pt, /content/drive/MyDrive/isic_2024_second_place/tmp/inference_temp/ids_0828-resnext50-transV2-lr1e-3-target_decay001-bs32_8-ep200-neg3-cluster7t5-tsgkf.npy
Configuration file for 0827-tip_finetune_onlyImage-swin_tiny_scratch_l2d192h8-tabV3-transV8-lr5e-4-warmup50-bs_64_8-neg50-ep200-tsgkf-lr1e-3-warmup5-bs_64_2-transV8-ep80 not found at /content/drive/MyDrive/isic_2024_second_place/working/configs/experiment/0827-tip_finetune_onlyImage-swin_tiny_scratch_l2d192h8-tabV3-transV8-lr5e-4-warmup50-bs_64_8-neg50-ep200-tsgkf-lr1e-3-warmup5-bs_64_2-transV8-ep80.yaml. Skipping this experiment.
Prediction files for 0825-tip_finetune_onlyImage-convnextv2_nano_scratch_l2d192h8-tabV3-transV8-lr5e-4-warmup50-bs_64_8-neg

100%|██████████| 1042/1042 [02:23<00:00,  7.26it/s]


Columns in oof_forecasts_eva_base.parquet: ['isic_id', 'target', 'patient_id', 'age_approx', 'sex', 'anatom_site_general', 'clin_size_long_diam_mm', 'image_type', 'tbp_tile_type', 'tbp_lv_A', 'tbp_lv_Aext', 'tbp_lv_B', 'tbp_lv_Bext', 'tbp_lv_C', 'tbp_lv_Cext', 'tbp_lv_H', 'tbp_lv_Hext', 'tbp_lv_L', 'tbp_lv_Lext', 'tbp_lv_areaMM2', 'tbp_lv_area_perim_ratio', 'tbp_lv_color_std_mean', 'tbp_lv_deltaA', 'tbp_lv_deltaB', 'tbp_lv_deltaL', 'tbp_lv_deltaLB', 'tbp_lv_deltaLBnorm', 'tbp_lv_eccentricity', 'tbp_lv_location', 'tbp_lv_location_simple', 'tbp_lv_minorAxisMM', 'tbp_lv_nevi_confidence', 'tbp_lv_norm_border', 'tbp_lv_norm_color', 'tbp_lv_perimeterMM', 'tbp_lv_radial_color_std_max', 'tbp_lv_stdL', 'tbp_lv_stdLExt', 'tbp_lv_symm_2axis', 'tbp_lv_symm_2axis_angle', 'tbp_lv_x', 'tbp_lv_y', 'tbp_lv_z', 'attribution', 'copyright_license', 'lesion_id', 'iddx_full', 'iddx_1', 'iddx_2', 'iddx_3', 'iddx_4', 'iddx_5', 'mel_mitotic_index', 'mel_thick_mm', 'tbp_lv_dnn_lesion_confidence', 'hdf5_key', 't

Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric perio

0.19553257653138129
